In [ ]:
!pip install transformers torch scikit-learn pandas numpy

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, EarlyStoppingCallback
import torch

In [ ]:
data = pd.read_csv("SentenceEntityCombinedSentimentDataset.csv")

In [ ]:
#removing null values
data = data[data['sentiment']!="none"] 
# in this dataset we get null values when sentiment is none so removing that
print(data.head(10))

In [ ]:
# Y- output
y = data['sentiment']
print("Output: \n",y[:10])

In [ ]:
# label encoding 
Sentiment_mappings = {
    "Positive" : 2,
    "Neutral" : 1,
    "Negative": 0, 
    "Mixed": 1
}

In [ ]:
y_mapped = pd.DataFrame(y.map(Sentiment_mappings).values, columns=["Sentiment"])

In [ ]:
print(y_mapped)

In [ ]:
# Load FinBERT and tokenizer
model_name = "yiyanghkust/finbert-tone"
tokenizer = AutoTokenizer.from_pretrained(model_name)
sentimentModel = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 3)

In [ ]:
encodings = tokenizer(
    data["sentence"].tolist(),
    data["entity"].tolist(),
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt"
)

In [ ]:
encodings["input_ids"].shape  # should be (num_samples, 128)

In [ ]:
train_texts, temp_texts, train_entities, temp_entities, y_train, y_temp = train_test_split(data["sentence"].tolist(),
                                                                                                     data["entity"].tolist(), 
                                                                                                     y_mapped, 
                                                                                                     test_size=0.3,
                                                                                                     random_state = 42,
                                                                                                     stratify=y_mapped
                                                                                                    )

In [ ]:
val_texts, test_texts, val_entities, test_entities, y_val, y_test = train_test_split(temp_texts,
                                                                                               temp_entities,
                                                                                               y_temp,
                                                                                               test_size = 0.5,
                                                                                               random_state = 42,
                                                                                               stratify=y_temp
                                                                                              )

In [ ]:
train_encodings = tokenizer(train_texts, train_entities, truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(val_texts, val_entities, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, test_entities, truncation=True, padding=True, max_length=128)

In [ ]:
train_labels = torch.tensor(y_train.values)
val_labels = torch.tensor(y_val.values)
test_labels = torch.tensor(y_test.values)


In [ ]:
from torch.utils.data import Dataset

class FinDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]))
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FinDataset(train_encodings, y_train.values)
val_dataset = FinDataset(val_encodings, y_val.values)
test_dataset = FinDataset(test_encodings, y_test.values)
#now randomly we are splitting if accuracy is less use index splitting so that all entities belonging to a sentence fall in one category.

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1) # for confidence scores remove argmax
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')  # weighted handles class imbalance
    return {"accuracy": acc, "f1": f1}


In [ ]:

training_args = TrainingArguments(
    output_dir="./finbert_results",
    eval_strategy="steps",  
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    logging_steps=50,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
)


In [ ]:

trainer = Trainer(
    model=sentimentModel,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],  # stop if no improvement in 3 evals
)


In [ ]:
trainer.train()

In [ ]:
trainer.evaluate(test_dataset)

In [ ]:
predictions = trainer.predict(test_dataset)
pred_labels = predictions.predictions.argmax(-1)

from sklearn.metrics import classification_report
print(classification_report(y_test, pred_labels, target_names=["Negative", "Neutral", "Positive"]))

In [ ]:
trainer.save_model("./finbert-entity-sentiment")
tokenizer.save_pretrained("./finbert-entity-sentiment")